# TAARDIS-27B — chat on a free Colab GPU

A 27B model whose **weights, head, embeddings, norms, scales *and* KV cache are ternary integers**, served by [taardis-llama.cpp](https://github.com/CodeMasterCody3D/taardis-llama.cpp). **Runtime → Change runtime type → GPU**, then run the cell below. It builds the fork (~3 min on A100/L4, ~35 min on a T4), picks the biggest configuration your GPU fits (1M-token ternary context first), and shows a link to the chat UI.

Model card: https://huggingface.co/CodeMasterCody3D/taardis-27b-full-ternary

In [ ]:

#@title TAARDIS-27B chat — full-ternary 27B with a ternary KV cache, on this Colab GPU { display-mode: "form" }
# Build ~3 min on A100/L4, ~35 min on a T4 (Colab gives T4 runtimes 2 vCPUs). Then a link appears.
import os, subprocess, time, json, urllib.request
def sh(cmd, timeout=None, check=True):
    print(f"$ {cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = (p.stdout or "") + (p.stderr or "")
    print(out[-2500:], flush=True)
    if check: assert p.returncode == 0, f"FAILED: {cmd}"
    return out

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip().splitlines()
assert gpu, "No GPU: Runtime -> Change runtime type -> GPU"
name, mem, cc = [x.strip() for x in gpu[0].split(",")]
vram_mb = int(mem.split()[0]); arch = cc.replace(".", "")
print(f"GPU: {name}  VRAM: {vram_mb} MiB  sm_{arch}  (x{len(gpu)})", flush=True)

if not os.path.exists("/content/fork/build/bin/llama-server"):
    sh("rm -rf /content/fork && git clone --depth 50 -b q1_0_g128-port https://github.com/CodeMasterCody3D/taardis-llama.cpp /content/fork && cd /content/fork && git log --oneline -1")
    print("building llama-server (this is the slow part)...", flush=True)
    sh(f"cmake -S /content/fork -B /content/fork/build -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON "
       f"-DCMAKE_CUDA_ARCHITECTURES={arch} -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null "
       f"&& cmake --build /content/fork/build -j $(nproc) --target llama-server 2>&1 | tail -2", timeout=5400)

from huggingface_hub import hf_hub_download
REPO = "CodeMasterCody3D/taardis-27b-full-ternary"
M = hf_hub_download(REPO, "TAARDIS-27B-Full-Ternary-V2-1.75bit.gguf", local_dir="/content")
L = hf_hub_download(REPO, "TAARDIS-27B-Doctors-V2.lora.gguf", local_dir="/content")

K1 = "-ctk q1_t_g128 -ctv q1_t_g128 -fa on"
if float(cc) >= 7.5:
    LADDER = [
        ("V2 + Doctors · k1 ternary KV · 1M ctx",   f"--lora {L} -c 1000000 {K1}"),
        ("V2 alone · k1 ternary KV · 1M ctx",       f"-c 1000000 {K1}"),
        ("V2 + Doctors · k1 ternary KV · 512k ctx", f"--lora {L} -c 524288 {K1}"),
        ("V2 + Doctors · k1 ternary KV · 256k ctx", f"--lora {L} -c 262144 {K1}"),
        ("V2 + Doctors · k1 ternary KV · 128k ctx", f"--lora {L} -c 131072 {K1}"),
        ("V2 + Doctors · q4_0 KV · 64k ctx",        f"--lora {L} -c 65536 -ctk q4_0 -ctv q4_0 -fa on"),
        ("V2 + Doctors · f16 KV · 16k ctx",         f"--lora {L} -c 16384"),
    ]
else:  # Pascal and older: no flash-attn -> f16 KV only
    LADDER = [("V2 + Doctors · f16 KV · 32k ctx (pre-Turing GPU)", f"--lora {L} -c 32768"),
              ("V2 + Doctors · f16 KV · 8k ctx", f"--lora {L} -c 8192")]

B = "/content/fork/build/bin/llama-server"
def start(flags):
    return subprocess.Popen(f"{B} -m {M} -ngl 99 --host 127.0.0.1 --port 8080 -t 2 --parallel 1 {flags}",
                            shell=True, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
def probe():
    req = urllib.request.Request("http://127.0.0.1:8080/v1/chat/completions",
        data=json.dumps({"messages": [{"role": "user", "content": "Say OK."}], "max_tokens": 8}).encode(),
        headers={"Content-Type": "application/json"})
    return urllib.request.urlopen(req, timeout=300).status == 200

srv = None; CONFIG = None; FLAGS = None
for label, flags in LADDER:
    print(f"\n--- trying: {label}", flush=True)
    p = start(flags); up = False
    for _ in range(96):                             # up to 8 min to load + allocate
        time.sleep(5)
        if p.poll() is not None: break
        try:
            if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5).status == 200:
                up = True; break
        except Exception: pass
    if up:
        try:
            if probe(): srv, CONFIG, FLAGS = p, label, flags; break
        except Exception as e: print("probe failed:", e, flush=True)
    print(open("/content/server.log").read()[-800:], flush=True)
    p.kill(); time.sleep(3)
assert srv, "no configuration fit this GPU -- see the logs above"
print(f"\nCONFIG_OK: {CONFIG}", flush=True)

from google.colab.output import eval_js
from IPython.display import HTML, display
url = eval_js("google.colab.kernel.proxyPort(8080)")
display(HTML(f"""
<div style="font-family:system-ui;padding:16px;border:2px solid #2a9d8f;border-radius:12px;max-width:720px">
  <div style="font-size:22px;font-weight:700">🟢 TAARDIS is up — <a href="{url}" target="_blank">open the chat</a></div>
  <div style="margin-top:8px;color:#444">Running: <b>{CONFIG}</b> on {name}.<br>
  Every weight and every KV-cache value is a ternary integer. Leave this cell running; stopping it stops the model.</div>
</div>"""))
print(url, flush=True)

t0 = time.time()
while True:                                        # keep-alive; revive the server if it dies
    time.sleep(60)
    if srv.poll() is not None:
        print("server exited -- restarting", flush=True); print(open("/content/server.log").read()[-1500:], flush=True)
        srv = start(FLAGS)
    if int(time.time() - t0) % 600 < 60:
        print(f"up {int((time.time()-t0)/60)} min · {CONFIG}", flush=True)
